In [77]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import os
import sys

import warnings
warnings.filterwarnings('ignore') # bo jakieś błedy wyskakują

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/Studia-II/Uczenie-Maszynowe/Lista5/Zad3

In [78]:
X_train_emb = np.load('../data/image_emb.npy', allow_pickle=True)
X_test_emb = np.load('../data/image_emb_test.npy', allow_pickle=True)

y_test_raw = np.load('../data/image_labels_test.npy', allow_pickle=True)
known_labels_data = np.load('../data/image_labels.npy', allow_pickle=True)

In [79]:
# etykiety tekstowe w danych treningowych
mask_known = np.array([isinstance(x, str) and x != '' for x in known_labels_data])
known_indices = np.where(mask_known)[0] 
known_true_classes_text = known_labels_data[mask_known]

In [80]:
print(f"Wszystkie mask_known: {len(mask_known)}, niezerowe: {np.count_nonzero(mask_known)}\n")
print(f"Znane indeksy: {known_indices}\n")
print(f"Znane klasy: {known_true_classes_text}")


Wszystkie mask_known: 42500, niezerowe: 50

Znane indeksy: [  447   582  1104  1527  2474  2640  3345  3694  4078  4797  6071  6960
  7344  8232 10108 12493 13020 13124 14168 14809 14993 15568 16736 16737
 20355 20892 21990 22167 23435 23841 27150 28148 28936 30671 31177 31413
 31414 33340 34371 34727 35474 35764 36478 36940 37218 38255 38260 38373
 38708 41791]

Znane klasy: ['truck' 'automobile' 'dog' 'deer' 'truck' 'automobile' 'airplane' 'ship'
 'horse' 'dog' 'airplane' 'frog' 'cat' 'automobile' 'frog' 'bird'
 'airplane' 'bird' 'bird' 'dog' 'cat' 'frog' 'deer' 'ship' 'bird' 'cat'
 'cat' 'dog' 'airplane' 'airplane' 'deer' 'dog' 'automobile' 'ship'
 'automobile' 'frog' 'deer' 'truck' 'truck' 'horse' 'truck' 'frog' 'bird'
 'ship' 'horse' 'ship' 'cat' 'horse' 'deer' 'horse']


In [81]:
# unikalne etykiety tekstowe z obu zbiorów
all_unique_labels = np.unique(np.concatenate((known_true_classes_text, y_test_raw)))

k = len(all_unique_labels)
print(f"\nWykryto {k} unikalnych klas: {all_unique_labels}")


Wykryto 10 unikalnych klas: ['airplane' 'automobile' 'bird' 'cat' 'deer' 'dog' 'frog' 'horse' 'ship'
 'truck']


In [82]:
# mapping tekst -> id
label_to_int_map = {label: i for i, label in enumerate(all_unique_labels)}

# mapping labelsów na liczby
known_true_classes = np.array([label_to_int_map[label] for label in known_true_classes_text])
y_test = np.array([label_to_int_map[label] for label in y_test_raw])

In [83]:
kmeans = KMeans(n_clusters=k, random_state=2137, n_init='auto') # auto to też 10, ale z random init
train_cluster_labels = kmeans.fit_predict(X_train_emb)

clusters_of_known_photos = train_cluster_labels[known_indices] # jaki klaster przyporządkowaliśmy odpowiedniej rzeczy
cluster_to_class_map = {} 

# teraz na podstawie tego co wylądował w tym samym klastrze będziemy decydować co ten klaster reprezetnuje
# jeśli dominują np ptaki, to robimy, że ten klaster reprezetnuje klasę ptaków

for cluster_id in range(k):
    true_classes_in_this_cluster = known_true_classes[clusters_of_known_photos == cluster_id]
    most_common_class = np.bincount(true_classes_in_this_cluster).argmax()
    cluster_to_class_map[cluster_id] = most_common_class

    print(f"Klaster {cluster_id}, dominanta: {most_common_class}")

Klaster 0, dominanta: 0
Klaster 1, dominanta: 2
Klaster 2, dominanta: 6
Klaster 3, dominanta: 3
Klaster 4, dominanta: 0
Klaster 5, dominanta: 9
Klaster 6, dominanta: 8
Klaster 7, dominanta: 5
Klaster 8, dominanta: 7
Klaster 9, dominanta: 1


In [84]:
test_cluster_labels = kmeans.predict(X_test_emb)
y_pred = [cluster_to_class_map.get(cluster, 0) for cluster in test_cluster_labels]
y_pred = np.array(y_pred)

In [85]:
accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=all_unique_labels)
conf_matrix = confusion_matrix(y_test, y_pred)

In [86]:
print(f"\n--- WYNIKI EWALUACJI ---")
print(f"Dokładność (Accuracy): {accuracy * 100:.2f}%")
print("\nRaport Klasyfikacji (z nazwami tekstowymi):")
print(report)
print("\nMacierz Pomyłek:")
print(conf_matrix)

# By definition a confusion matrix :math:`C` is such that :math:`C_{i, j}`
#     is equal to the number of observations known to be in group :math:`i` and
#     predicted to be in group :math:`j`.


--- WYNIKI EWALUACJI ---
Dokładność (Accuracy): 72.59%

Raport Klasyfikacji (z nazwami tekstowymi):
              precision    recall  f1-score   support

    airplane       0.84      0.95      0.89       692
  automobile       0.95      0.94      0.94       759
        bird       0.49      0.75      0.59       761
         cat       0.36      0.92      0.52       746
        deer       0.00      0.00      0.00       761
         dog       0.91      0.72      0.80       758
        frog       0.97      0.75      0.85       754
       horse       0.96      0.51      0.67       760
        ship       0.97      0.87      0.92       739
       truck       0.98      0.88      0.93       770

    accuracy                           0.73      7500
   macro avg       0.74      0.73      0.71      7500
weighted avg       0.74      0.73      0.71      7500


Macierz Pomyłek:
[[657   3   3  20   0   0   1   0   8   0]
 [  9 712   0  25   0   0   0   1   2  10]
 [ 35   0 567 150   0   2   6   0   

Jak na jedynie 50 znanych opisów, wygląda to dla mnie nieźle, z seedem 42 było np 76 %